In [74]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rakibulhasanshaon69/the-verdict-txt")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\patil\.cache\kagglehub\datasets\rakibulhasanshaon69\the-verdict-txt\versions\1


In [75]:
import os

# Join the directory path with the actual filename
file_path = os.path.join(path, 'the-verdict.txt')  # Replace with your actual file name

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

print("Total number of characters in the file:", len(content))
print(content[:99])  # Print the first 500 characters of the file

Total number of characters in the file: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [76]:
import re

text = "Hello, world! This is a test string with punctuation."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', ' ', 'string', ' ', 'with', ' ', 'punctuation.']


In [77]:
import re
preprocessed = [item for item in re.split(r'([.,!?;()]|--|\s+)', content) if item.strip()]
preprocessed = [item.split() for item in preprocessed if item.strip()]
print(preprocessed[:10])  # Print the first 10 elements of the preprocessed list

[['I'], ['HAD'], ['always'], ['thought'], ['Jack'], ['Gisburn'], ['rather'], ['a'], ['cheap'], ['genius']]


In [78]:
all_words = sorted(set([word for sublist in preprocessed for word in sublist]))
vocab_size = len(all_words)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1207


Create a vocabulary using the-verdict.txt file

In [79]:
vocab = {token:integer for integer, token in enumerate(all_words)}

## Each token or unique word is given a token Id.

In [80]:
for i, item in enumerate(vocab.items()):
    if i < 50:  # Print only the first 10 items
        print(item)

('!', 0)
('"', 1)
('"Ah', 2)
('"Be', 3)
('"Begin', 4)
('"By', 5)
('"Come', 6)
('"Destroyed', 7)
('"Don\'t', 8)
('"Gisburns"', 9)
('"Grindles', 10)
('"Hang', 11)
('"Has', 12)
('"How', 13)
('"I', 14)
('"I\'d', 15)
('"If', 16)
('"It', 17)
('"It\'s', 18)
('"Jack', 19)
('"Money\'s', 20)
('"Moon-dancers"', 21)
('"Mr', 22)
('"Mrs', 23)
('"My', 24)
('"Never', 25)
('"Of', 26)
('"Oh', 27)
('"Once', 28)
('"Only', 29)
('"Or', 30)
('"That', 31)
('"The', 32)
('"Then', 33)
('"There', 34)
('"There:', 35)
('"This', 36)
('"We', 37)
('"Well', 38)
('"What', 39)
('"When', 40)
('"Why', 41)
('"Yes', 42)
('"You', 43)
('"but', 44)
('"deadening', 45)
('"dragged', 46)
('"effects"', 47)
('"interesting":', 48)
('"lift', 49)


In [81]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inverse_vocab = {v: k for k, v in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([.,!?;()]|--|\s+)', text)
        tokens = [token for token in tokens if token.strip()]
        return [self.vocab[token] for token in tokens if token in self.vocab]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([.,!?;()])', r'\1', text)
        return text

In [82]:
tokenizer = SimpleTokenizer(vocab)
text = "Hello, world! This is a test string with punctuation."
ids = tokenizer.encode(text)
print("Encoded IDs:", ids)

Encoded IDs: [64, 0, 652, 180, 1182, 66]


In [83]:
text = "Bravo, do you like tea?"
print(tokenizer.encode(text))  # Encode the text into ID

[64, 418, 1201, 698, 1045, 68]


## Adding Special Context Tokens

In [84]:
# Flatten preprocessed if it contains sublists
if preprocessed and isinstance(preprocessed[0], list):
    preprocessed = [token for sublist in preprocessed for token in sublist]

# 1. Deduplicate and sort your unique tokens
all_tokens = sorted(set(preprocessed))

# 2. Add special tokens ONLY if they aren't already present
for special_token in ['<UNK>', '<endoftext>']:
    if special_token not in all_tokens:
        all_tokens.append(special_token)

# 3. Build vocabulary mapping
vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [85]:
class simpleTokenizerv2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inverse_vocab = {v: k for k, v in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([.,!?;()]|--|\s+)', text)
        tokens = [token for token in tokens if token.strip()]
        return [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]

    def decode(self, ids):
        text = " ".join([self.inverse_vocab.get(i, '<UNK>') for i in ids])
        text = re.sub(r'\s+([.,!?;()])', r'\1', text)
        return text

In [87]:
tokenizer = simpleTokenizerv2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <endoftext> ".join((text1, text2))
print(text)

Hello, do you like tea? <endoftext> In the sunlit terraces of the palace.


In [88]:
tokenizer.encode(text)

[1207,
 64,
 418,
 1201,
 698,
 1045,
 68,
 1208,
 109,
 1057,
 1028,
 1053,
 792,
 1057,
 1207,
 66]

In [89]:
tokenizer.decode(tokenizer.encode(text))

'<UNK>, do you like tea? <endoftext> In the sunlit terraces of the <UNK>.'

## The major problem with word based tokenizer is that we need large vocabulary the problem with character level tokenizer is that the meaning of the word is lost in the process.

## So the best option comes is the Subword based tokenization.